# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Lets list all the record sets contained in the schema, along with fields (columns) for each record set.

In [ ]:
# List all record sets with their @id and their fields with @id
from pprint import pprint

record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
    print(f"Number of record sets: {len(record_sets)}\n")
else:
    # Sometimes record_sets can be in `recordSet` field instead
    record_sets = getattr(metadata, 'recordSet', [])
    print(f"Number of record sets: {len(record_sets)}\n")

record_set_ids = []
for i, rs in enumerate(record_sets):
    print(f"Record set {i+1} @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs}")
    record_set_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    record_set_ids.append(record_set_id)
    # Attempt to get fields/columns for this record set
    if isinstance(rs, dict):
        print("  Fields (columns):")
        fields = rs.get('field', rs.get('fields', []))
        for field in fields:
            if isinstance(field, dict) and '@id' in field:
                print(f"    - {field['@id']}")
            else:
                print(f"    - {field}")
    else:
        # Try to extract using dataset.metadata
        rs_obj = None
        try:
            rs_obj = dataset.metadata[record_set_id]
        except Exception:
            pass
        if rs_obj is not None and hasattr(rs_obj, 'fields'):
            print("  Fields (columns):")
            for field in rs_obj.fields:
                if hasattr(field, '@id'):
                    print(f"    - {field['@id']}")
                else:
                    print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Note: Insert the `@id` of the record set you wish to analyze. You may update the list of record sets if more are found._

In [ ]:
# If record sets or their IDs are not clearly listed, you may need to look them up via the Croissant schema or previous exploration.
# For this dataset, we inspect likely options; otherwise you can provide the @id(s).

# Example record set IDs for this dataset; supply your actual @ids from section 2
record_sets = [
    'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # main proteomics table
]

dataframes = {}

for record_set in record_sets:
    print(f"Loading records for record set {record_set} ...")
    records = list(dataset.records(record_set=record_set))
    if len(records) == 0:
        print(f"No records found for {record_set}.")
    else:
        dataframes[record_set] = pd.DataFrame.from_records(records)
        print(f"Loaded {len(dataframes[record_set])} rows, columns:", dataframes[record_set].columns.tolist())

# Pick one for further analysis
main_rs_id = record_sets[0]
df = dataframes[main_rs_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

_We will select a numeric field, for example, the molecular weight ("MW" or similar), and perform filtering and normalization._

In [ ]:
# Identify a numeric field (e.g. molecular weight, peptide count, or intensities). Update this as appropriate for your dataset.
# For demonstration, we'll assume a field ID like 'MW' or similar ('MW' as column name).

numeric_field = None
for col in df.columns:
    # Common molecular weight field variations
    if col.lower() in ['mw', 'molecular_weight', 'protein_mw']:
        numeric_field = col
        break
# If not found, try another quantitative field, such as 'abundance' or 'count'
if numeric_field is None:
    for col in df.columns:
        if 'abundance' in col.lower() or 'count' in col.lower() or 'number' in col.lower():
            numeric_field = col
            break

if numeric_field is not None:
    # Ensure the field is numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)  # E.g., top quartile filtering
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a string/categorical field (e.g., 'Description', 'Sample', 'Protein')
    group_field = None
    for col in df.columns:
        if col.lower() in ['description', 'protein', 'accession', 'sample', 'uniprot']:
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No group field identified for aggregation.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

_Example: Histogram of a numeric field, or scatter plot if two relevant fields are available._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Optionally, scatter plot vs. another field
    # Find second numeric field
    other_numeric = None
    for c in df.columns:
        if c != numeric_field and pd.api.types.is_numeric_dtype(df[c]):
            other_numeric = c
            break
    if other_numeric:
        plt.figure(figsize=(8,6))
        sns.scatterplot(x=df[numeric_field], y=df[other_numeric])
        plt.xlabel(numeric_field)
        plt.ylabel(other_numeric)
        plt.title(f"Scatter: {numeric_field} vs. {other_numeric}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR2-compliant Croissant dataset using its schema URL and explored its structure with `mlcroissant`.
- Main quantitative fields (such as molecular weight or abundance) were identified for basic exploratory data analysis.
- Filtering, normalization, group aggregation, and simple visualizations provided insights into the protein mass spectrometry data.
- For deeper analysis, consult the full Croissant metadata for field definitions and apply more advanced methods such as multi-dimensional scaling, clustering, or protein annotation cross-referencing.